Рекомендательная система фильмов

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

In [27]:
ratings_full_df = pd.read_csv('ratings.csv')
movies_df = pd.read_csv('movies.csv')

In [29]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
62418,209157,We (2018),Drama
62419,209159,Window of the Soul (2001),Documentary
62420,209163,Bad Poems (2018),Comedy|Drama
62421,209169,A Girl Thing (2001),(no genres listed)


In [31]:
ratings_full_df

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510
...,...,...,...,...
25000090,162541,50872,4.5,1240953372
25000091,162541,55768,2.5,1240951998
25000092,162541,56176,2.0,1240950697
25000093,162541,58559,4.0,1240953434


In [33]:
ratings_cut_df = ratings_full_df[:200000]
ratings_cut_df

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510
...,...,...,...,...
199995,1409,48516,4.0,1287843475
199996,1409,48696,4.0,1287854818
199997,1409,48738,3.5,1287854109
199998,1409,48774,4.0,1287849077


In [35]:
mean_user_ratings = ratings_full_df.groupby("userId")["rating"].mean()
mean_user_ratings

userId
1         3.814286
2         3.630435
3         3.697409
4         3.378099
5         3.752475
            ...   
162537    4.039604
162538    3.415584
162539    4.510638
162540    3.829545
162541    3.365385
Name: rating, Length: 162541, dtype: float64

In [37]:
ratings_cut_df = ratings_cut_df.merge(mean_user_ratings, how="left", on="userId", suffixes=("", "user_mean"))
ratings_cut_df

#Денормализация данных, для одного пользователя появилось много строк

,userId,movieId,rating,timestamp,ratinguser_mean
0,1,296,5.0,1147880044,3.814286
1,1,306,3.5,1147868817,3.814286
2,1,307,5.0,1147868828,3.814286
3,1,665,5.0,1147878820,3.814286
4,1,899,3.5,1147868510,3.814286
...,...,...,...,...,...
199995,1409,48516,4.0,1287843475,3.043624
199996,1409,48696,4.0,1287854818,3.043624
199997,1409,48738,3.5,1287854109,3.043624
199998,1409,48774,4.0,1287849077,3.043624


In [39]:
mean_movie_ratings = ratings_full_df.groupby("movieId")["rating"].mean()
mean_movie_ratings

movieId
1         3.893708
2         3.251527
3         3.142028
4         2.853547
5         3.058434
            ...   
209157    1.500000
209159    3.000000
209163    4.500000
209169    3.000000
209171    3.000000
Name: rating, Length: 59047, dtype: float64

In [41]:
movies_df = movies_df.merge(mean_movie_ratings, how="left", on="movieId").rename({"rating": "rating_movie_mean"}, axis=1)
movies_df

,movieId,title,genres,rating_movie_mean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.893708
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.251527
2,3,Grumpier Old Men (1995),Comedy|Romance,3.142028
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.853547
4,5,Father of the Bride Part II (1995),Comedy,3.058434
...,...,...,...,...
62418,209157,We (2018),Drama,1.500000
62419,209159,Window of the Soul (2001),Documentary,3.000000
62420,209163,Bad Poems (2018),Comedy|Drama,4.500000
62421,209169,A Girl Thing (2001),(no genres listed),3.000000


In [43]:
ratings_cut_df

,userId,movieId,rating,timestamp,ratinguser_mean
0,1,296,5.0,1147880044,3.814286
1,1,306,3.5,1147868817,3.814286
2,1,307,5.0,1147868828,3.814286
3,1,665,5.0,1147878820,3.814286
4,1,899,3.5,1147868510,3.814286
...,...,...,...,...,...
199995,1409,48516,4.0,1287843475,3.043624
199996,1409,48696,4.0,1287854818,3.043624
199997,1409,48738,3.5,1287854109,3.043624
199998,1409,48774,4.0,1287849077,3.043624


In [45]:
datetimes = pd.to_datetime(ratings_cut_df["timestamp"], unit='s')


In [47]:
datetimes

0        2006-05-17 15:34:04
1        2006-05-17 12:26:57
2        2006-05-17 12:27:08
3        2006-05-17 15:13:40
4        2006-05-17 12:21:50
                 ...        
199995   2010-10-23 14:17:55
199996   2010-10-23 17:26:58
199997   2010-10-23 17:15:09
199998   2010-10-23 15:51:17
199999   2010-10-23 14:26:35
Name: timestamp, Length: 200000, dtype: datetime64[ns]

In [49]:
ratings_cut_df['year'] = datetimes.dt.year
ratings_cut_df['month'] = datetimes.dt.month
ratings_cut_df['day'] = datetimes.dt.day
ratings_cut_df['day_of_week'] = datetimes.dt.day_of_week
ratings_cut_df['hour'] = datetimes.dt.hour

def get_part_of_day(hour):
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 17:
        return 'afternoon'
    elif 17 <= hour < 21:
        return 'evening'
    else:
        return 'night'

ratings_cut_df['part_of_day'] = ratings_cut_df['hour'].apply(get_part_of_day) #Применяем функцию get_part.... чтобы уменьшить количество входных данных
#для модели, делаем по сути из числового признака категориальный

ratings_cut_df = pd.get_dummies(ratings_cut_df, columns=['part_of_day'], prefix='time') #OneHot, добавление новых 4 колонок

ratings_cut_df = ratings_cut_df.drop(columns=['timestamp'])  #Выбрасываем timestamp, он нам уже не нужен

ratings_cut_df

,userId,movieId,rating,ratinguser_mean,year,month,day,day_of_week,hour,time_afternoon,time_evening,time_morning,time_night
0,1,296,5.0,3.814286,2006,5,17,2,15,True,False,False,False
1,1,306,3.5,3.814286,2006,5,17,2,12,True,False,False,False
2,1,307,5.0,3.814286,2006,5,17,2,12,True,False,False,False
3,1,665,5.0,3.814286,2006,5,17,2,15,True,False,False,False
4,1,899,3.5,3.814286,2006,5,17,2,12,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,1409,48516,4.0,3.043624,2010,10,23,5,14,True,False,False,False
199996,1409,48696,4.0,3.043624,2010,10,23,5,17,False,True,False,False
199997,1409,48738,3.5,3.043624,2010,10,23,5,17,False,True,False,False
199998,1409,48774,4.0,3.043624,2010,10,23,5,15,True,False,False,False


In [53]:
genres_list = movies_df['genres'].apply(lambda x : x.split('|'))
genres_list


0        [Adventure, Animation, Children, Comedy, Fantasy]
1                           [Adventure, Children, Fantasy]
2                                        [Comedy, Romance]
3                                 [Comedy, Drama, Romance]
4                                                 [Comedy]
                               ...                        
62418                                              [Drama]
62419                                        [Documentary]
62420                                      [Comedy, Drama]
62421                                 [(no genres listed)]
62422                           [Action, Adventure, Drama]
Name: genres, Length: 62423, dtype: object

In [57]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

mlb.fit(genres_list)

list(mlb.classes_)

['(no genres listed)',
 'Action',
 'Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'IMAX',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western']

In [59]:
mlb.transform(genres_list)

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0]])

In [61]:
genres_encoded = pd.DataFrame(mlb.transform(genres_list), columns=mlb.classes_, index=movies_df.index)

In [63]:
genres_encoded.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [65]:
movies_encoded = pd.concat([movies_df.drop("genres", axis=1), genres_encoded], axis=1)
movies_encoded.head()

,movieId,title,rating_movie_mean,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),3.893708,0,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),3.251527,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),3.142028,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),2.853547,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),3.058434,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [67]:
df = ratings_cut_df.merge(movies_encoded, how="left", on="movieId")

In [69]:
df

,userId,movieId,rating,ratinguser_mean,year,month,day,day_of_week,hour,time_afternoon,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,296,5.0,3.814286,2006,5,17,2,15,True,...,0,0,0,0,0,0,0,1,0,0
1,1,306,3.5,3.814286,2006,5,17,2,12,True,...,0,0,0,0,0,0,0,0,0,0
2,1,307,5.0,3.814286,2006,5,17,2,12,True,...,0,0,0,0,0,0,0,0,0,0
3,1,665,5.0,3.814286,2006,5,17,2,15,True,...,0,0,0,0,0,0,0,0,1,0
4,1,899,3.5,3.814286,2006,5,17,2,12,True,...,0,0,0,1,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,1409,48516,4.0,3.043624,2010,10,23,5,14,True,...,0,0,0,0,0,0,0,1,0,0
199996,1409,48696,4.0,3.043624,2010,10,23,5,17,False,...,0,0,0,0,0,1,0,0,0,0
199997,1409,48738,3.5,3.043624,2010,10,23,5,17,False,...,0,0,0,0,0,0,0,1,0,0
199998,1409,48774,4.0,3.043624,2010,10,23,5,15,True,...,0,0,0,0,0,0,1,1,0,0


In [73]:
mean_genres_by_users = pd.concat(
    [
        df["userId"],
        df[list(mlb.classes_)].multiply(df["rating"], axis=0)
    ], axis=1
).groupby("userId").mean()

mean_genres_by_users

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
userId,,,,,,,,,,,,,,,,,,,,
1,0.000000,0.235714,0.585714,0.114286,0.164286,1.271429,0.464286,0.028571,2.928571,0.257143,0.050000,0.050000,0.000000,0.264286,0.200000,1.071429,0.264286,0.307143,0.278571,0.035714
2,0.000000,1.326087,1.592391,0.334239,0.497283,1.135870,0.307065,0.000000,1.766304,0.627717,0.000000,0.065217,0.154891,0.195652,0.160326,0.584239,0.611413,0.625000,0.279891,0.067935
3,0.005335,1.853659,1.113567,0.303354,0.271341,0.926829,0.782012,0.014482,1.375762,0.444360,0.032774,0.243140,0.461128,0.032774,0.355945,0.323171,1.262195,1.345274,0.146341,0.045732
4,0.000000,1.909091,1.448347,0.444215,0.371901,1.208678,0.607438,0.084711,0.758264,0.456612,0.000000,0.130165,0.411157,0.105372,0.283058,0.142562,1.111570,0.820248,0.123967,0.105372
5,0.000000,0.663366,0.801980,0.148515,0.297030,1.732673,0.574257,0.000000,1.702970,0.277228,0.000000,0.138614,0.118812,0.257426,0.326733,0.702970,0.445545,0.950495,0.089109,0.148515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,0.000000,0.388448,0.297848,0.074745,0.135900,1.556059,0.593431,0.073613,2.151755,0.235561,0.106455,0.148358,0.006795,0.245753,0.272933,0.919592,0.177803,0.652322,0.199320,0.074745
1406,0.000000,0.409091,0.409091,0.454545,0.454545,1.590909,1.318182,0.000000,2.500000,0.000000,0.000000,0.409091,0.000000,0.000000,0.545455,0.590909,0.409091,1.000000,0.863636,0.000000
1407,0.000000,0.539062,0.562500,0.132812,0.320312,2.914062,0.398438,0.023438,0.859375,0.351562,0.015625,0.125000,0.000000,0.343750,0.187500,0.953125,0.304688,0.343750,0.195312,0.156250


In [75]:
df = df.merge(mean_genres_by_users, how='left', on="userId", suffixes=("", "mean_gen"))
df

,userId,movieId,rating,ratinguser_mean,year,month,day,day_of_week,hour,time_afternoon,...,Film-Noirmean_gen,Horrormean_gen,IMAXmean_gen,Musicalmean_gen,Mysterymean_gen,Romancemean_gen,Sci-Fimean_gen,Thrillermean_gen,Warmean_gen,Westernmean_gen
0,1,296,5.0,3.814286,2006,5,17,2,15,True,...,0.050000,0.050000,0.000000,0.264286,0.200000,1.071429,0.264286,0.307143,0.278571,0.035714
1,1,306,3.5,3.814286,2006,5,17,2,12,True,...,0.050000,0.050000,0.000000,0.264286,0.200000,1.071429,0.264286,0.307143,0.278571,0.035714
2,1,307,5.0,3.814286,2006,5,17,2,12,True,...,0.050000,0.050000,0.000000,0.264286,0.200000,1.071429,0.264286,0.307143,0.278571,0.035714
3,1,665,5.0,3.814286,2006,5,17,2,15,True,...,0.050000,0.050000,0.000000,0.264286,0.200000,1.071429,0.264286,0.307143,0.278571,0.035714
4,1,899,3.5,3.814286,2006,5,17,2,12,True,...,0.050000,0.050000,0.000000,0.264286,0.200000,1.071429,0.264286,0.307143,0.278571,0.035714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,1409,48516,4.0,3.043624,2010,10,23,5,14,True,...,0.020499,0.068627,0.045455,0.059715,0.199643,0.514260,0.340463,0.899287,0.386809,0.176471
199996,1409,48696,4.0,3.043624,2010,10,23,5,17,False,...,0.020499,0.068627,0.045455,0.059715,0.199643,0.514260,0.340463,0.899287,0.386809,0.176471
199997,1409,48738,3.5,3.043624,2010,10,23,5,17,False,...,0.020499,0.068627,0.045455,0.059715,0.199643,0.514260,0.340463,0.899287,0.386809,0.176471
199998,1409,48774,4.0,3.043624,2010,10,23,5,15,True,...,0.020499,0.068627,0.045455,0.059715,0.199643,0.514260,0.340463,0.899287,0.386809,0.176471


In [79]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(["rating", "title"], axis=1), df["rating"], test_size=0.2, random_state=42)

In [87]:
def naive_approach(X_train, y_mean, movieId):
    movie_data = X_train[X_train["movieId"]==movieId]
    if movie_data.empty:
        return y_mean
    else:
        return movie_data.iloc[0]["rating_movie_mean"]

In [89]:
y_mean = y_train.mean()
predict = [naive_approach(X_train, y_mean, movieId) for movieId in X_test["movieId"]]

predict

[3.5701620505454055,
 3.7648009950248755,
 3.30687887203589,
 2.992050660199407,
 3.6987526802388584,
 4.413576004516335,
 3.453398738612474,
 4.151341616415071,
 2.9386574074074074,
 3.4711328976034856,
 3.6686106507395477,
 2.972902097902098,
 3.995328608247423,
 3.2634129456559364,
 4.1805253650920005,
 4.237947624243627,
 4.114713689799014,
 3.6224948492227007,
 2.748033877797943,
 4.284353213163313,
 3.5569092991773403,
 3.817136507193824,
 4.014331931413782,
 3.586023142509135,
 3.4105008077544428,
 3.6222502851556135,
 3.185682326621924,
 3.8968066922200353,
 3.508920477209269,
 3.3279432064681522,
 3.44905273937532,
 4.04407206509105,
 2.993827160493827,
 3.2194312796208533,
 3.8756302521008403,
 2.886744743146127,
 4.129097007632781,
 3.183039195863782,
 4.0903399807075225,
 3.859743262873914,
 4.212267265284564,
 2.6973419250906163,
 3.804476930105071,
 3.356494275562574,
 3.6402393970740357,
 3.379296875,
 3.332691196301291,
 3.182581786030062,
 4.077005923532579,
 3.1024590

In [95]:
def get_recommendation(X_train, y_mean, userId, movieId):
    movie_data = X_train[X_train["movieId"]==movieId]
    if movie_data.empty:
        return y_mean

    user_data = X_test[X_test["userId"]==userId].iloc[0]
    movie_mean_rating = movie_data.iloc[0]["rating_movie_mean"]
    user_mean_rating = user_data["ratinguser_mean"]

    final_score = (0.5 * movie_mean_rating) + (0.5 * user_mean_rating)
    return final_score


In [97]:
predict = [get_recommendation(X_train, y_mean, userId, movieId) for userId, movieId in zip(X_test["userId"], X_test["movieId"])]
predict

[3.283288910577362,
 3.7415554270899025,
 2.988074852684612,
 2.9649142189885924,
 3.48639557088866,
 4.288788002258167,
 3.7627163184587795,
 3.819314876004146,
 3.426603530493542,
 3.766563753383953,
 3.6646624682269167,
 3.102752187711066,
 3.927664304123711,
 3.381706472827968,
 3.919210050967053,
 4.007403564187929,
 3.7585402768521696,
 3.6444176790144818,
 3.0400685994155765,
 3.9003066878824697,
 3.418042750275169,
 3.545587484366143,
 3.8099915185324438,
 3.366966555177397,
 3.558753588590597,
 4.061125142577807,
 3.2588928238275674,
 3.8821094163975576,
 3.8556999080261223,
 3.210003349265822,
 3.6818011964774984,
 3.814893175402668,
 3.3549382716049383,
 3.4584199081031093,
 3.6742021123517903,
 2.913886800682223,
 3.9395485038163907,
 2.9478414370123507,
 3.9674672876510586,
 3.8022854245404054,
 3.904744743753393,
 3.3322152663427764,
 3.639307430569777,
 3.7859454270503385,
 3.4364208372970344,
 3.3059495762600166,
 3.197654009365599,
 3.0231794379066717,
 3.8776875597277